In [7]:
!python.exe -m pip install --upgrade pip

   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 1.8/1.8 MB 19.6 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.2
    Uninstalling pip-24.2:
      Successfully uninstalled pip-24.2


In [1]:
%pip install copulas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
%pip install plotly
%pip install --upgrade nbformat

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [1]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
import scipy.stats as stats

def plot_cdf_pdf_plotly():
    # Generate 10000 evenly distributed values from -4 to 4
    x = np.linspace(-4.0, 4.0, 10000)

    # Compute their Probability Densities and Cumulative Distributions
    pdf = stats.norm.pdf(x)
    cdf = stats.norm.cdf(x)

    fig = make_subplots(rows=1, cols=2, subplot_titles=("PDF", "CDF"))

    fig.add_trace(
        go.Scatter(x=x, y=pdf),
        row=1, col=1
    )
    fig.update_xaxes(title_text="x", row=1, col=1)
    fig.update_yaxes(title_text="f(x)", row=1, col=1)

    fig.add_trace(
        go.Scatter(x=x, y=cdf),
        row=1, col=2
    )
    fig.update_xaxes(title_text="x", row=1, col=2)
    fig.update_yaxes(title_text="F(x)", row=1, col=2)

    # Update yaxis properties

    fig.update_layout(height=400, width=900, showlegend=False)
    fig.show()

plot_cdf_pdf_plotly()

**CDF Formula:**
$$F(x) = \int_{-\infty}^{x} f(t) \, dt$$

Cumulative Distribution Function $F$ can be defined as random variable $Y$, so, Y distribution is uniform distribution(0,1). it is core of Copula, It make it possible to compare between different distributions.

In [3]:
from scipy import stats

X = stats.norm.rvs(size=10000)
X_pit = stats.norm.cdf(X)

fig = make_subplots(rows=1, cols=2, subplot_titles=("Samples", "Transformed Samples"))

fig.add_trace(
    go.Histogram(x=X),
    row=1, col=1
)

fig.add_trace(
    go.Histogram(x=X_pit),
    row=1, col=2
)

fig.update_layout(height=400, width=900, showlegend=False)
fig.show()

The key intuition underlying copula functions is the idea that marginal distributions can be modeled independently from the joint distribution.

In [4]:
from copulas.datasets import sample_bivariate_age_income

df = sample_bivariate_age_income()
df.head()

,age,income
0,48.935913,399.161393
1,39.234323,364.225531
2,55.659901,406.475105
3,31.810637,341.276022
4,65.342336,414.347815


In [5]:
from copulas.visualization import scatter_2d

scatter_2d(df)

In [8]:
from copulas.visualization import dist_1d

dist1 = dist_1d(df['age'], title='Age')

dist1.show()

dist_1d(df['income'], title='Income')

Limitations of the Traditional Approach:
- When age follows a normal distribution and income follows a gamma distribution, modeling their joint distribution simultaneously becomes extremely complex.

The Copula Approach:
- Each variable's marginal distribution is modeled independently
- The dependency structure (correlation) is captured separately in the uniform distribution space
- Via inverse transformation, samples from any target distribution can be generated

In [9]:
from copulas.datasets import sample_univariate_beta

data = sample_univariate_beta()

from copulas.visualization import dist_1d

dist_1d(data)

In [10]:
from copulas.univariate import BetaUnivariate

beta = BetaUnivariate()
beta.fit(data)

beta._params

c:\Users\jaeseok\AppData\Local\Programs\Python\Python312\Lib\site-packages\scipy\stats\_continuous_distns.py:785: RuntimeWarning:

invalid value encountered in sqrt



{'loc': 4.052009283524141,
 'scale': 0.947975527152111,
 'a': 2.552323190906777,
 'b': 0.9041091899047261}

In [19]:
sampled = beta.sample(10000)

In [20]:
from copulas.visualization import compare_1d

compare_1d(data, sampled)

# 히스토그램을 부드럽게 연결한 KDE(Kernel Density Estimation)이다.

In [21]:
probability_density = beta.pdf(sampled)

In [24]:
probability_density[0:5]

array([1.60777227, 1.79962071, 1.89001001, 2.40413416, 1.43758059])

In [28]:
import pandas as pd

pd.options.plotting.backend = "plotly"
pd.DataFrame({
    'data': sampled,
    'probability_density': probability_density
}).sort_values('data').set_index('data').plot()

In [29]:
cumulative_distribution = beta.cumulative_distribution(sampled)

In [30]:
pd.DataFrame({
    'data': sampled,
    'cumulative distribution': cumulative_distribution
}).sort_values('data').set_index('data').plot()

In [31]:
parameters = beta.to_dict()

In [32]:
parameters

{'loc': 4.052009283524141,
 'scale': 0.947975527152111,
 'a': 2.552323190906777,
 'b': 0.9041091899047261,
 'type': 'copulas.univariate.beta.BetaUnivariate'}

In [33]:
from copulas.univariate import Univariate

new_beta = Univariate.from_dict(parameters)

In [34]:
new_sampled = new_beta.sample(1000)

compare_1d(data, new_sampled)

In [35]:
univariate = Univariate()
univariate.fit(data)

c:\Users\jaeseok\AppData\Local\Programs\Python\Python312\Lib\site-packages\scipy\stats\_continuous_distns.py:785: RuntimeWarning:

invalid value encountered in sqrt

c:\Users\jaeseok\AppData\Local\Programs\Python\Python312\Lib\site-packages\scipy\stats\_continuous_distns.py:785: RuntimeWarning:

invalid value encountered in sqrt



In [36]:
parameters = univariate.to_dict()

In [37]:
parameters

{'loc': 4.052009283524141,
 'scale': 0.947975527152111,
 'a': 2.552323190906777,
 'b': 0.9041091899047261,
 'type': 'copulas.univariate.beta.BetaUnivariate'}

In [38]:
from copulas.datasets import sample_univariates

data = sample_univariates()

In [39]:
data.head()

,bernoulli,bimodal,uniform,normal,degenerate,exponential,beta
0,0.0,11.399355,0.498160,1.496714,0.37454,3.469268,4.796025
1,0.0,10.924634,2.802857,0.861736,0.37454,6.010121,4.935189
2,0.0,10.059630,1.927976,1.647689,0.37454,4.316746,4.637677
3,0.0,9.353063,1.394634,2.523030,0.37454,3.912943,4.945320
4,1.0,-0.234153,-0.375925,0.765847,0.37454,3.169625,4.726815


In [40]:
synth_data = pd.DataFrame()
distributions = []

for column in data.columns:
    real_data = data[column]
    univariate = Univariate()
    univariate.fit(real_data)
    synth_data[column] = univariate.sample(len(real_data))
    distributions.append(univariate.to_dict()['type'])

c:\Users\jaeseok\AppData\Local\Programs\Python\Python312\Lib\site-packages\scipy\stats\_continuous_distns.py:785: RuntimeWarning:

invalid value encountered in sqrt

c:\Users\jaeseok\AppData\Local\Programs\Python\Python312\Lib\site-packages\scipy\stats\_continuous_distns.py:790: RuntimeWarning:

The iteration is not making good progress, as measured by the 
  improvement from the last ten iterations.

c:\Users\jaeseok\AppData\Local\Programs\Python\Python312\Lib\site-packages\scipy\stats\_continuous_distns.py:790: RuntimeWarning:

The iteration is not making good progress, as measured by the 
  improvement from the last ten iterations.

c:\Users\jaeseok\AppData\Local\Programs\Python\Python312\Lib\site-packages\scipy\stats\_continuous_distns.py:785: RuntimeWarning:

invalid value encountered in sqrt

c:\Users\jaeseok\AppData\Local\Programs\Python\Python312\Lib\site-packages\scipy\stats\_continuous_distns.py:790: RuntimeWarning:

The iteration is not making good progress, as measured by t

In [41]:
distributions

['copulas.univariate.gaussian_kde.GaussianKDE',
 'copulas.univariate.gaussian_kde.GaussianKDE',
 'copulas.univariate.beta.BetaUnivariate',
 'copulas.univariate.gaussian_kde.GaussianKDE',
 'copulas.univariate.beta.BetaUnivariate',
 'copulas.univariate.beta.BetaUnivariate',
 'copulas.univariate.beta.BetaUnivariate']

In [42]:
for column in data.columns:
    fig = compare_1d(data[column], synth_data[column])
    fig.show()

# Multivariate Distributions

In [43]:
from copulas.datasets import sample_trivariate_xyz

data = sample_trivariate_xyz()

In [44]:
from copulas.visualization import scatter_3d

scatter_3d(data)

In [46]:
from copulas.multivariate import GaussianMultivariate
from copulas.visualization import compare_3d

dist = GaussianMultivariate()
dist.fit(data)

sampled = dist.sample(1000)

compare_3d(data, sampled)


c:\Users\jaeseok\AppData\Local\Programs\Python\Python312\Lib\site-packages\scipy\stats\_continuous_distns.py:785: RuntimeWarning:

invalid value encountered in sqrt

c:\Users\jaeseok\AppData\Local\Programs\Python\Python312\Lib\site-packages\scipy\stats\_continuous_distns.py:785: RuntimeWarning:

invalid value encountered in sqrt

c:\Users\jaeseok\AppData\Local\Programs\Python\Python312\Lib\site-packages\scipy\stats\_continuous_distns.py:785: RuntimeWarning:

invalid value encountered in sqrt

c:\Users\jaeseok\AppData\Local\Programs\Python\Python312\Lib\site-packages\scipy\stats\_continuous_distns.py:6435: RuntimeWarning:

divide by zero encountered in power

